# spellchk: default program

In [16]:
import pandas as pd
import numpy as np

In [17]:
# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

### Anlysis on Edit Distance

In [18]:
# Distribution of edit distances
print(f"\nEdit Distance Distribution:")
edit_dist_counts = df['Edit_Distance'].value_counts().sort_index()
for dist, count in edit_dist_counts.items():
    print(f"  Distance {dist}: {count} occurrences")


Edit Distance Distribution:
  Distance 1: 1200 occurrences
  Distance 2: 403 occurrences
  Distance 3: 88 occurrences
  Distance 4: 24 occurrences
  Distance 5: 7 occurrences
  Distance 6: 1 occurrences
  Distance 7: 1 occurrences


### Analysis on Transformer Score

In [19]:
# Number of N/A in Transformer Score
from regex import P


num_na = df['Transformer_Score'].isna().sum()
total_entries = len(df)
na_percentage = (num_na / total_entries) * 100
print(f"\n\nTRANSFORMER SCORE N/A COUNT: {num_na} out of {total_entries} ({na_percentage:.2f}%)")

# How many ground truths were found in top 10, top 20, etc.
print(f"\nGround Truth Rank Distribution:")
if df['Rank'].notna().any():
    in_top_5 = (df['Rank'] <= 5).sum()
    in_top_10 = (df['Rank'] <= 10).sum()
    in_top_20 = (df['Rank'] <= 20).sum()
    in_top_50 = (df['Rank'] <= 50).sum()
    in_top_100 = (df['Rank'] <= 100).sum()
    in_top_150 = (df['Rank'] <= 150).sum()
    in_top_200 = (df['Rank'] <= 200).sum()
    
    print(f"  In Top 5: {in_top_5} ({in_top_5/total_entries*100:.2f}%)")
    print(f"  In Top 10: {in_top_10} ({in_top_10/total_entries*100:.2f}%)")
    print(f"  In Top 20: {in_top_20} ({in_top_20/total_entries*100:.2f}%)")
    print(f"  In Top 50: {in_top_50} ({in_top_50/total_entries*100:.2f}%)")
    print(f"  In Top 100: {in_top_100} ({in_top_100/total_entries*100:.2f}%)")
    print(f"  In Top 150: {in_top_150} ({in_top_150/total_entries*100:.2f}%)")
    print(f"  In Top 200: {in_top_200} ({in_top_200/total_entries*100:.2f}%)")
else:
    print(f"  Not Found (N/A): {num_na} ({na_percentage:.2f}%)")




TRANSFORMER SCORE N/A COUNT: 402 out of 1724 (23.32%)

Ground Truth Rank Distribution:
  In Top 5: 768 (44.55%)
  In Top 10: 911 (52.84%)
  In Top 20: 1022 (59.28%)
  In Top 50: 1156 (67.05%)
  In Top 100: 1246 (72.27%)
  In Top 150: 1295 (75.12%)
  In Top 200: 1322 (76.68%)


In [24]:
# ============================================================================
# ANALYSIS 1: All Incorrect Predictions
# ============================================================================
print("\n" + "="*80)
print("INCORRECT PREDICTIONS")
print("="*80)

incorrect_df = df[df['Selected_Word'].str.lower() != df['Ground_Truth'].str.lower()]
num_incorrect = len(incorrect_df)
incorrect_percentage = (num_incorrect / total_entries) * 100

print(f"\nTotal Incorrect: {num_incorrect} out of {total_entries} ({incorrect_percentage:.2f}%)")
print(f"Total Correct: {total_entries - num_incorrect} ({100 - incorrect_percentage:.2f}%)")

if num_incorrect > 0:
    print("\n" + incorrect_df[['Typo', 'Ground_Truth', 'Selected_Word', 'Transformer_Score', 
                                'Rank', 'Edit_Distance', 'Similarity_Score']].to_string())


INCORRECT PREDICTIONS

Total Incorrect: 452 out of 1724 (26.22%)
Total Correct: 1272 (73.78%)

                         Typo              Ground_Truth    Selected_Word  Transformer_Score   Rank  Edit_Distance  Similarity_Score
1                      didm't                    didn't              did                NaN    NaN              1          0.833333
4                         Wat                      What              Wat           0.044161    1.0              1          0.857143
10                       Rjel                    Rachel               He                NaN    NaN              3          0.600000
11                  uhclaspig                unclasping         clapping                NaN    NaN              2          0.842105
28                   imprtont                 important         implicit                NaN    NaN              2          0.823529
33              simple-seeing            simple-seeming         simplest                NaN    NaN              

In [23]:
# ============================================================================
# ANALYSIS 2: Incorrect Predictions where Ground Truth was Available
# ============================================================================
print("\n\n" + "="*80)
print("INCORRECT PREDICTIONS (Ground Truth WAS in Top-100)")
print("="*80)

incorrect_with_score_df = df[
    (df['Selected_Word'].str.lower() != df['Ground_Truth'].str.lower()) & 
    (df['Transformer_Score'].notna())
]
num_incorrect_with_score = len(incorrect_with_score_df)
incorrect_with_score_percentage = (num_incorrect_with_score / total_entries) * 100

print(f"\nIncorrect (ground truth available): {num_incorrect_with_score} out of {total_entries} ({incorrect_with_score_percentage:.2f}%)")
print(f"Incorrect (ground truth NOT available): {num_incorrect - num_incorrect_with_score}")

if num_incorrect_with_score > 0:
    print("\n" + incorrect_with_score_df[['Typo', 'Ground_Truth', 'Selected_Word', 
                                          'Transformer_Score', 'Rank', 'Edit_Distance', 
                                          'Similarity_Score']].to_string())



INCORRECT PREDICTIONS (Ground Truth WAS in Top-100)

Incorrect (ground truth available): 50 out of 1724 (2.90%)
Incorrect (ground truth NOT available): 402

           Typo Ground_Truth Selected_Word  Transformer_Score   Rank  Edit_Distance  Similarity_Score
4           Wat         What           Wat           0.044161    1.0              1          0.857143
40      CChtios    Catholics       Parties           0.332139    1.0              5          0.500000
63        Theee        These         There           0.068274    3.0              1          0.800000
73           ad          and            as           0.002769   31.0              1          0.800000
75          col         cool          cold           0.003976   40.0              1          0.857143
138         ffm         from           for           0.105097    4.0              2          0.571429
243           t          not            to           0.573339    1.0              2          0.500000
246   crrntrues    centur